# Publication summary of $-F_A(Q^2)$

This notebook turns selected joint PROfit posterior chains into median $-F_A$ curves and pointwise 68% credible bands. Edit only the **Figure controls** cell for normal use, then run all cells. The gray band is the one selected prior; the lower panel shows every result relative to that prior's median.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

start = Path.cwd().resolve()
NOTEBOOK_DIR = next(
    (path / 'python' / 'scripts' for path in (start, *start.parents)
     if (path / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate python/scripts/postfit_physical_parameters.py')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from postfit_physical_parameters import SPECS, load_fit, plot_fa_summary

## Figure controls

Set a fit to `True` to draw its posterior. `COMPARISON_PRIOR` must be one key from the same list, or `None` for the conventional $M_A=1.014$ GeV dipole denominator. The comparison prior does not need to be enabled as a posterior.

In [ ]:
# Data selection
SUITE = 'nuwro_fit_results'  # NuWro: nuwro_fit_results; Asimov: asimov_fit_results
SHOW_POSTFIT = {
    'minerva_k8': False,
    'minerva_k7': False,
    'minerva_k6': True,
    'lqcd_k6': False,
    'minerva_lqcd_k6': False,
    "minerva_k6_uniform": False
}
COMPARISON_PRIOR = 'minerva_k6'  # one key above, or None

# Chain and plot controls
BURN_IN = 0
THIN = 1
Q2_RANGE = (0.01, 2.0)
X_SCALE = 'log'  # 'log' or 'linear'
RATIO_ZOOM = (0.01, 0.25, 0.90, 1.10)  # (Q2 min, Q2 max, ratio min, ratio max), or None
MAX_CURVES = 50_000

# Publication output
SAVE_FIGURE = True
OUTPUT_STEM = NOTEBOOK_DIR.parents[1] / 'figs' / SUITE / 'fa_postfit_summary'
SAVE_FORMATS = ('pdf', 'png')
PNG_DPI = 600

In [ ]:
selected = [key for key, enabled in SHOW_POSTFIT.items() if enabled]
requested = set(selected) | ({COMPARISON_PRIOR} if COMPARISON_PRIOR else set())
specs = {spec.key: spec for spec in SPECS}
unknown = requested - set(specs)
if unknown:
    raise ValueError(f'Unknown fit keys: {sorted(unknown)}')
if not requested:
    raise ValueError('Enable at least one posterior or choose a comparison prior.')

results = {}
for key in requested:
    result = load_fit(specs[key], SUITE, burn_in=BURN_IN, thin=THIN)
    if result is None:
        raise FileNotFoundError(f'No unique PROfile ROOT file for {key!r} in {SUITE!r}')
    results[key] = result
    print(f'{key}: {len(result["samples"]):,} posterior samples')

In [ ]:
figure = plot_fa_summary(
    results,
    show=selected,
    comparison_prior=COMPARISON_PRIOR,
    q2_range=Q2_RANGE,
    max_samples=MAX_CURVES,
    ratio_zoom=RATIO_ZOOM,
    xscale=X_SCALE,
)
display(figure)

if SAVE_FIGURE:
    OUTPUT_STEM.parent.mkdir(parents=True, exist_ok=True)
    for extension in SAVE_FORMATS:
        path = OUTPUT_STEM.with_suffix('.' + extension.lstrip('.'))
        kwargs = dict(bbox_inches='tight', pad_inches=.03, facecolor='white')
        if extension.lower().lstrip('.') == 'png':
            kwargs['dpi'] = PNG_DPI
        figure.savefig(path, **kwargs)
        print(f'Saved {path}')
plt.close(figure)

## Open-data summary

Reproduce the summary above with the open-data posterior chains and the same fit selection and plotting controls.

In [ ]:
OPEN_DATA_SUITE = 'opendata_fit_results'

open_data_results = {}
for key in requested:
    result = load_fit(specs[key], OPEN_DATA_SUITE, burn_in=BURN_IN, thin=THIN)
    if result is None:
        raise FileNotFoundError(
            f'No unique PROfile ROOT file for {key!r} in {OPEN_DATA_SUITE!r}'
        )
    open_data_results[key] = result
    print(f'{key}: {len(result["samples"]):,} posterior samples')

open_data_figure = plot_fa_summary(
    open_data_results,
    show=selected,
    comparison_prior=COMPARISON_PRIOR,
    q2_range=Q2_RANGE,
    max_samples=MAX_CURVES,
    ratio_zoom=RATIO_ZOOM,
    xscale=X_SCALE,
)
display(open_data_figure)

if SAVE_FIGURE:
    open_data_output_stem = (
        NOTEBOOK_DIR.parents[1] / 'figs' / OPEN_DATA_SUITE / 'fa_postfit_summary'
    )
    open_data_output_stem.parent.mkdir(parents=True, exist_ok=True)
    for extension in SAVE_FORMATS:
        path = open_data_output_stem.with_suffix('.' + extension.lstrip('.'))
        kwargs = dict(bbox_inches='tight', pad_inches=.03, facecolor='white')
        if extension.lower().lstrip('.') == 'png':
            kwargs['dpi'] = PNG_DPI
        open_data_figure.savefig(path, **kwargs)
        print(f'Saved {path}')
plt.close(open_data_figure)

## Asimov-data summary

Reproduce the summary above with the Asimov posterior chains and the same fit selection and plotting controls.

In [ ]:
ASIMOV_DATA_SUITE = 'asimov_fit_results'

asimov_data_results = {}
for key in requested:
    result = load_fit(specs[key], ASIMOV_DATA_SUITE, burn_in=BURN_IN, thin=THIN)
    if result is None:
        raise FileNotFoundError(
            f'No unique PROfile ROOT file for {key!r} in {ASIMOV_DATA_SUITE!r}'
        )
    asimov_data_results[key] = result
    print(f'{key}: {len(result["samples"]):,} posterior samples')

asimov_data_figure = plot_fa_summary(
    asimov_data_results,
    show=selected,
    comparison_prior=COMPARISON_PRIOR,
    q2_range=Q2_RANGE,
    max_samples=MAX_CURVES,
    ratio_zoom=RATIO_ZOOM,
    xscale=X_SCALE,
)
display(asimov_data_figure)

if SAVE_FIGURE:
    asimov_data_output_stem = (
        NOTEBOOK_DIR.parents[1] / 'figs' / ASIMOV_DATA_SUITE / 'fa_postfit_summary'
    )
    asimov_data_output_stem.parent.mkdir(parents=True, exist_ok=True)
    for extension in SAVE_FORMATS:
        path = asimov_data_output_stem.with_suffix('.' + extension.lstrip('.'))
        kwargs = dict(bbox_inches='tight', pad_inches=.03, facecolor='white')
        if extension.lower().lstrip('.') == 'png':
            kwargs['dpi'] = PNG_DPI
        asimov_data_figure.savefig(path, **kwargs)
        print(f'Saved {path}')
plt.close(asimov_data_figure)